# Module 4 Slides 5–10 Demo
## Amazon Bedrock Prompt Management with reusable templates, variables, personas, business-unit prompts, centralized control, and versions

This notebook is designed for a trainer-led class in VS Code or Jupyter.

The default `RUN_AWS = False` mode performs a safe local demonstration. It renders templates and shows the governance metadata without creating AWS resources or invoking a model. Set `RUN_AWS = True` only when the class account has the required permissions and the selected model is available in the configured Region.

The AWS section demonstrates:

- a reusable managed prompt template;
- runtime variables using `{{variable_name}}`;
- a fixed persona for each business unit;
- centralized naming, ownership metadata, and tags;
- immutable prompt versions;
- optional invocation of a versioned prompt with the Converse API.

**Important architecture point:** Prompt Management stores templates, variables, configurations, and versions. It does not by itself implement approval routing, user authorization, demographic fairness measurement, or production traffic allocation.


## 1. Configuration

For AWS mode, install or update `boto3`, configure credentials, and grant only the actions needed for the demo. Typical actions include prompt create, get, list, update, version, and delete operations plus model invocation when `INVOKE_VERSIONED_PROMPT` is enabled.

Use a classroom sandbox. The notebook creates uniquely named resources so it does not overwrite existing prompts.


In [ ]:
RUN_AWS = False   # Local preview by default. Set True to create real Prompt Management resources.
INVOKE_VERSIONED_PROMPT = True
DELETE_DEMO_RESOURCES = True

AWS_REGION = "<AWS_REGION>"  # a Region where the chosen model is enabled
MODEL_ID = "amazon.nova-lite-v1:0"  # Replace if this model is unavailable in the class Region
PROMPT_PREFIX = "dd300-module4"

TEMPERATURE = 0.2
MAX_TOKENS = 500

print("Mode:", "AWS" if RUN_AWS else "LOCAL PREVIEW")
print("Region:", AWS_REGION)
print("Model or inference profile:", MODEL_ID)


## 2. Central persona catalog

The persona catalog is the governance source used to build the managed prompts. Each persona states its audience, expertise, communication style, and boundaries. The model does not receive permission merely because the persona says that it has permission; real authorization remains outside the prompt.


In [ ]:
PERSONAS = {
    "customer_support": {
        "display_name": "Customer Support Specialist",
        "audience": "customers with mixed technical experience",
        "expertise": "product usage, account guidance, and first-line troubleshooting",
        "style": "clear, calm, concise, and non-technical unless the customer asks for technical detail",
        "boundaries": "Do not invent account facts. Do not expose internal notes. Escalate requests that require privileged account actions.",
        "owner": "CustomerExperience",
    },
    "sales_engineering": {
        "display_name": "Sales Engineering Advisor",
        "audience": "technical evaluators and solution architects",
        "expertise": "architecture trade-offs, integration requirements, security assumptions, and proof-of-concept planning",
        "style": "technically precise, evidence-based, and explicit about assumptions",
        "boundaries": "Do not promise unsupported features, prices, compliance status, or delivery dates.",
        "owner": "SalesEngineering",
    },
    "executive_briefing": {
        "display_name": "Executive Briefing Advisor",
        "audience": "senior business leaders",
        "expertise": "business impact, risk, cost, dependencies, and decisions",
        "style": "brief, outcome-focused, and free of unnecessary implementation detail",
        "boundaries": "Separate confirmed facts from assumptions. Do not hide material risks or uncertainty.",
        "owner": "ExecutivePrograms",
    },
}

for key, persona in PERSONAS.items():
    print(f"{key:20} owner={persona['owner']:20} audience={persona['audience']}")


## 3. Reusable template and variables

Every business unit uses the same template structure. The fixed persona text changes by governed business unit. Runtime data enters only through the variables `audience`, `context`, and `question`.

In Amazon Bedrock Prompt Management, variables appear inside double curly braces. Values are supplied at invocation time through `promptVariables`.


In [ ]:
BASE_TEMPLATE = '''
Role
You are the {{persona_name}}.

Audience
{{persona_audience}}

Expertise
{{persona_expertise}}

Communication style
{{persona_style}}

Boundaries
{{persona_boundaries}}

Runtime audience
{{audience}}

Approved context
{{context}}

User question
{{question}}

Instructions
1. Use only the approved context for organization-specific claims.
2. If required information is missing, state the gap and ask one focused clarification question.
3. Match the response depth and terminology to the persona and runtime audience.
4. Clearly label assumptions.
5. Return a concise answer followed by an optional next step.
'''.strip()

RUNTIME_VARIABLES = ["audience", "context", "question"]

def persona_template(persona):
    replacements = {
        "{{persona_name}}": persona["display_name"],
        "{{persona_audience}}": persona["audience"],
        "{{persona_expertise}}": persona["expertise"],
        "{{persona_style}}": persona["style"],
        "{{persona_boundaries}}": persona["boundaries"],
    }
    rendered = BASE_TEMPLATE
    for placeholder, value in replacements.items():
        rendered = rendered.replace(placeholder, value)
    return rendered

MANAGED_TEMPLATES = {unit: persona_template(persona) for unit, persona in PERSONAS.items()}
print(MANAGED_TEMPLATES["customer_support"])


## 4. Local persona comparison

This step does not call a model. It shows how the same runtime values are inserted into different governed personas. Ask the class to compare the expected depth, vocabulary, and boundaries.


In [ ]:
SAMPLE_INPUT = {
    "audience": "A customer evaluating whether the service fits a production workload",
    "context": "The approved architecture uses API Gateway, Lambda, and Amazon Bedrock. The pilot target is 500 requests per minute. No production SLA has been approved.",
    "question": "Can this solution support our production launch next month?",
}

def render_runtime(template, values):
    result = template
    for name, value in values.items():
        result = result.replace("{{" + name + "}}", str(value))
    return result

for unit in PERSONAS:
    print("\n" + "=" * 90)
    print(unit.upper())
    print("=" * 90)
    print(render_runtime(MANAGED_TEMPLATES[unit], SAMPLE_INPUT))


## 5. Centralized prompt registry

The registry shows the fields an organization should track even when Prompt Management stores the prompt resource: owner, purpose, variables, model, default configuration, lifecycle status, and evaluation suite. Tags assist discovery and cost allocation, but tags do not replace an approval workflow.


In [ ]:
PROMPT_REGISTRY = []
for unit, persona in PERSONAS.items():
    PROMPT_REGISTRY.append({
        "logical_name": f"{PROMPT_PREFIX}-{unit}",
        "business_unit": unit,
        "owner": persona["owner"],
        "purpose": "Respond to approved business questions using an audience-specific persona",
        "variables": RUNTIME_VARIABLES,
        "model": MODEL_ID,
        "temperature": TEMPERATURE,
        "status": "TRAINING_DEMO",
        "evaluation_suite": "module4-persona-regression-v1",
    })

for item in PROMPT_REGISTRY:
    print(item)


## 6. Build the Bedrock Prompt Management request

The API currently accepts one saved prompt variant configuration per prompt resource. The console can compare temporary variants before one is saved as the draft. This demo creates one managed prompt resource per fixed business-unit persona. A separate application could instead use one prompt with a runtime `persona` variable, but fixed resources provide clearer ownership and version history for the class exercise.


In [ ]:
import json
import time

RUN_SUFFIX = time.strftime("%Y%m%d-%H%M%S", time.gmtime())

def build_prompt_variant(unit, template_text):
    return {
        "name": "default",
        "modelId": MODEL_ID,
        "templateType": "TEXT",
        "templateConfiguration": {
            "text": {
                "text": template_text,
                "inputVariables": [{"name": name} for name in RUNTIME_VARIABLES],
            }
        },
        "inferenceConfiguration": {
            "text": {
                "temperature": TEMPERATURE,
                "maxTokens": MAX_TOKENS,
                "topP": 0.9,
            }
        },
        "metadata": [
            {"key": "business_unit", "value": unit},
            {"key": "course_module", "value": "DD300-M04"},
        ],
    }

CREATE_REQUESTS = {}
for unit, persona in PERSONAS.items():
    name = f"{PROMPT_PREFIX}-{unit}-{RUN_SUFFIX}".replace("_", "-")[:100]
    CREATE_REQUESTS[unit] = {
        "name": name,
        "description": f"Training demo prompt for {unit}; owner {persona['owner']}"[:200],
        "defaultVariant": "default",
        "variants": [build_prompt_variant(unit, MANAGED_TEMPLATES[unit])],
        "tags": {
            "course": "DD300",
            "module": "M04",
            "business-unit": unit.replace("_", "-"),
            "purpose": "training-demo",
        },
    }

print(json.dumps(CREATE_REQUESTS["customer_support"], indent=2))


## 7. Create the managed prompts and Version 1

This cell performs AWS writes only when `RUN_AWS=True`. Each created prompt begins as `DRAFT`; `CreatePromptVersion` takes an immutable snapshot suitable for application use.


In [ ]:
CREATED = {}

if not RUN_AWS:
    print("Preview only. No AWS prompt resources were created.")
else:
    import boto3
    from botocore.exceptions import ClientError

    agent_client = boto3.client("bedrock-agent", region_name=AWS_REGION)
    runtime_client = boto3.client("bedrock-runtime", region_name=AWS_REGION)

    for unit, request in CREATE_REQUESTS.items():
        created = agent_client.create_prompt(**request)
        prompt_id = created["id"]
        version = agent_client.create_prompt_version(
            promptIdentifier=prompt_id,
            description="Version 1: initial governed persona template",
        )
        CREATED[unit] = {
            "prompt_id": prompt_id,
            "draft_arn": created["arn"],
            "v1": version["version"],
            "v1_arn": version["arn"],
            "name": request["name"],
        }
        print(unit, CREATED[unit])


## 8. Update the drafts and create Version 2

Version 2 adds an explicit evidence requirement. Version 1 remains available and unchanged. This is the core versioning lesson: applications should reference a numbered version ARN, not the mutable draft.


In [ ]:
V2_ADDITION = '''

Evidence requirement
For every organization-specific statement, identify the supporting sentence from the approved context. If the context does not support the statement, do not make the claim.
'''.rstrip()

V2_TEMPLATES = {unit: text + V2_ADDITION for unit, text in MANAGED_TEMPLATES.items()}

if not RUN_AWS:
    print("Preview of the Version 2 change:\n", V2_ADDITION)
else:
    for unit, item in CREATED.items():
        request = CREATE_REQUESTS[unit]
        agent_client.update_prompt(
            promptIdentifier=item["prompt_id"],
            name=request["name"],
            description=request["description"],
            defaultVariant="default",
            variants=[build_prompt_variant(unit, V2_TEMPLATES[unit])],
        )
        version = agent_client.create_prompt_version(
            promptIdentifier=item["prompt_id"],
            description="Version 2: require evidence for organization-specific claims",
        )
        item["v2"] = version["version"]
        item["v2_arn"] = version["arn"]
        print(unit, "Version 1:", item["v1_arn"], "Version 2:", item["v2_arn"])


## 9. Inspect centralized resources and versions

Use `GetPrompt` for the draft or a numbered version and `ListPrompts` with a prompt identifier to list versions. In the console, open Prompt Management and compare the prompt versions visually.


In [ ]:
if not RUN_AWS:
    print("Expected lifecycle: DRAFT → Version 1 → edit DRAFT → Version 2")
    print("Application recommendation: reference the numbered version ARN, not DRAFT.")
else:
    for unit, item in CREATED.items():
        draft = agent_client.get_prompt(promptIdentifier=item["prompt_id"])
        versions = agent_client.list_prompts(promptIdentifier=item["prompt_id"])
        print("\n", unit)
        print("Draft updated at:", draft.get("updatedAt"))
        for summary in versions.get("promptSummaries", []):
            print("  version=", summary.get("version"), "arn=", summary.get("arn"))


## 10. Optional: invoke a numbered managed prompt

The version ARN becomes the `modelId` for the Converse API. Do not send `messages` or `system` when invoking a managed prompt because the template is already stored in Prompt Management. Supply only the declared `promptVariables`.

Model invocation incurs cost and requires the chosen model or inference profile to be available and permitted in the Region.


In [ ]:
if not RUN_AWS or not INVOKE_VERSIONED_PROMPT:
    print("Invocation skipped. Enable both RUN_AWS and INVOKE_VERSIONED_PROMPT to call a model.")
else:
    for unit, item in CREATED.items():
        response = runtime_client.converse(
            modelId=item["v2_arn"],
            promptVariables={name: {"text": SAMPLE_INPUT[name]} for name in RUNTIME_VARIABLES},
        )
        text = response["output"]["message"]["content"][0]["text"]
        print("\n" + "=" * 80)
        print(unit.upper())
        print(text)


## 11. Classroom comparison and knowledge check

Ask learners to answer these questions before showing the suggested explanation:

1. **Why use variables?** They separate reusable instructions from runtime values and reduce copied prompt text.
2. **Why create separate business-unit prompts?** Fixed personas can have distinct owners, boundaries, tests, and version histories. One variable-driven prompt is also valid when shared governance is preferred.
3. **Why use numbered versions?** A version is an immutable snapshot. A draft is mutable and should not be the production deployment target.
4. **Does Prompt Management approve a prompt automatically?** No. An organization still needs review criteria and an approval workflow or release process.
5. **Does a persona enforce IAM permissions?** No. Persona boundaries guide model behavior; IAM and application authorization enforce access.
6. **Do compared prompt variants automatically receive production traffic?** No. Testing variants and production traffic allocation are separate concerns.
7. **Exam clue:** reusable templates, variables, variants, and versions point to Prompt Management. Harmful-content controls point to Guardrails. Durable service orchestration points to Step Functions.


## 12. Optional cleanup

Cleanup is deliberately disabled. Set `DELETE_DEMO_RESOURCES=True` only after confirming the exact resources in `CREATED`. The cell deletes the numbered versions created by this notebook and then deletes each draft prompt.


In [ ]:
if not RUN_AWS or not DELETE_DEMO_RESOURCES:
    print("Cleanup skipped. Set DELETE_DEMO_RESOURCES=True after reviewing CREATED.")
else:
    for unit, item in CREATED.items():
        for key in ["v1", "v2"]:
            if item.get(key):
                agent_client.delete_prompt(
                    promptIdentifier=item["prompt_id"],
                    promptVersion=item[key],
                )
                print("Deleted", unit, "version", item[key])
        agent_client.delete_prompt(promptIdentifier=item["prompt_id"])
        print("Deleted draft prompt", unit, item["prompt_id"])


## AWS references

- [Construct and store reusable prompts with Prompt Management](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management.html)
- [Create a prompt using Prompt Management](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management-create.html)
- [Run Prompt Management code samples](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management-code-ex.html)
- [Create prompt versions](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management-version-create.html)
- [Compare prompt versions](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management-version-compare.html)
